# Caveman: train an adapter for ultra-compressed answers
**Open Weight Room · Supervised fine-tuning**

Turn Julius Brussee's public [Caveman skill](https://github.com/JuliusBrussee/caveman/blob/2fd153c67988e980fb0b2455c90832159a6a5a25/skills/caveman/SKILL.md)
into an adapter with Fireworks Serverless Training, without provisioning GPUs.
The skill strips filler while aiming to preserve technical accuracy. In the author's words:
“All technical substance stay. Only fluff die.”

**What you'll do:** inspect answers generated with the skill, train on 2,348 prepared examples,
then compare your adapter with the original model and the model using the skill prompt.
The goal is to learn the skill's terse English style in the weights, so you can use it without the style prompt.

### Cost: about 4.68 USD to train your own copy using our data

The included dataset and selected settings let you skip data generation and the experiments we ran
to choose the recipe. The final training run used 526,162 training tokens over two epochs (74 steps).
At the study's recorded rate of 8.89 USD per million training tokens, that's **about 4.68 USD for training**.
Use your account's current rate when budgeting a new run.

| Your path with the included data | Estimated cost |
|---|---:|
| Explore the saved answers and charts | Free |
| Use our prepared training dataset | No generation cost |
| Train one copy with the selected settings | **4.68 USD at the study's rate** |
| Evaluate and try your copy | Additional inference and judging costs; not yet measured for this learner path |

### What we spent developing it

We generated and filtered the data, tried several training runs, and evaluated the results along the way.
You do not need to repeat that whole process to train from the included data.

| Development work | Our recorded cost |
|---|---:|
| Data generation and filtering | 20.80 USD |
| Five training runs, including the final one | 22.64 USD |
| Seven evaluations | 40.22 USD |
| **Total development and evaluation** | **84.76 USD** |

The final training run's 4.68 USD is already included in the training total above.
These estimates use the historical rates recorded during the study.

**Before you start:** follow [setup](../skill_to_weights/SETUP.md), then open Jupyter in this directory.
Live training needs a Fireworks account with model access and credits. **Run All is free by default:**
paid stages remain off until you explicitly enable them.

[Compare example answers](#see-it) · [Train your adapter](#reproduce-it) ·
[Understand the teaching data](#understand-it) · [Improve the data](#change-it)

<a id="see-it"></a>
## 1. Compare answers: does Caveman keep what matters?
Before training, let's look at what we're trying to teach. Below, three versions of the same model answer
one question using the same supplied facts:

- **Base:** the original model, without the skill.
- **Skill prompt:** the original model with Caveman's instructions.
- **Adapter:** the trained model, without Caveman's instructions.

Read the answers and ask: **what did Caveman remove, and did it keep the information needed to answer?**
These are saved answers, so this cell makes no API calls. The pass/fail labels show how our evaluator
scored each answer; we'll explain those checks after the comparison.

In [ ]:
from pathlib import Path
import json
from notebook_display import answers, figure
HERE = Path.cwd()
BUNDLE = HERE / "release/v2c"
recorded = json.loads((BUNDLE / "recorded.json").read_text())
fixtures = {r["id"]: r for r in map(json.loads, (BUNDLE / "caveman_test.jsonl").read_text().splitlines())}
answers(recorded["examples"][0])

### Did the shorter answers still answer the questions?
One example shows the style. The chart checks whether it holds across **392 questions excluded from training**.
For each question, we collected **eight separate answers from each model configuration** to measure variation.
Each draw starts fresh, without access to the previous answer.

The left panel shows the percentage passing every content and mechanical check. The right shows average
answer length in words. Higher quality and fewer words is the combination we want.
The bars' uncertainty intervals account for related questions and repeated answers.

*Evaluation note:* “pass” means the study's checks passed. This test set was used for two adapter versions,
so treat the comparison as exploratory. Word counts measure answer length. Billing uses token counts.

In [ ]:
figure(recorded["rows"], fixtures, title="Recorded adapter · TEST",
       path=HERE / "figures/caveman_v2c.png");

The recorded adapter passes more checks and uses fewer words than the skill prompt on this set.
These results describe this recipe and task distribution. Evaluate your own skill on representative tasks.
We will use the same checks on your run, and read the failures as well as the averages.

<a id="reproduce-it"></a>
## 2. Train your adapter: inspect the prepared data
The teaching pack is included so your first run does not require generating thousands of answers.
Each row pairs task information with a teacher answer that passed filtering. Training removes the neutral
system message as well as omitting the skill: the adapter must learn from the request itself.

In [ ]:
from collections import Counter
import hashlib
manifest = json.loads((BUNDLE / "manifest.json").read_text())
for name, expected in manifest["files"].items():
    assert hashlib.sha256((BUNDLE / name).read_bytes()).hexdigest() == expected, f"Changed bundle file: {name}"
training_rows = list(map(json.loads, (BUNDLE / "caveman_train.jsonl").read_text().splitlines()))
print(f"{len(training_rows):,} teaching answers")
print("Coverage:", dict(Counter(r["meta"]["family"] for r in training_rows)))
for message in training_rows[0]["messages"]:
    if message["role"] != "system":
        print(f"\n{message['role'].upper()}\n{message['content']}")

### Review the plan, then explicitly enable training
This cell prints the exact command before doing anything. The settings match the recorded adapter.
Rank was not isolated as the cause of improvement; data coverage and target selection also changed.

**Paid stage:** training only. It does not load an adapter for inference. Choose a new model name and run folder.
To repeat a completed cell, leave `RUN_TRAINING=False`; this cell refuses to reuse an existing run folder.

Two epochs give each selected answer two training passes. Rank 64 and the learning rate reproduce the selected run; this study did not establish that either is optimal. Keep them fixed while first testing a data change.


In [ ]:
import os, sys, subprocess, shlex
from getpass import getpass
RUN_TRAINING = False
OUTPUT_MODEL_ID = "YOUR_NEW_MODEL_NAME"
RUN_DIR = HERE / "artifacts/my-caveman"
command = [sys.executable, str(HERE / "run_pack_sft.py"),
    "--pack", str(BUNDLE / "caveman_train.jsonl"), "--drop-system-prompt",
    "--epochs", "2", "--batch-size", "64", "--lora-rank", "64", "--lora-alpha", "128",
    "--learning-rate", "0.0001", "--seed", "29",
    "--output-model-id", OUTPUT_MODEL_ID, "--run-dir", str(RUN_DIR)]
# Show a portable command; execute with this kernel's actual Python and directory.
def portable_command(args):
    return shlex.join(["python" if arg == sys.executable else str(arg).replace(str(HERE) + os.sep, "")
                      for arg in args])
print(portable_command(command))
if RUN_TRAINING:
    assert OUTPUT_MODEL_ID != "YOUR_NEW_MODEL_NAME", "Choose a new model name first."
    assert not RUN_DIR.exists(), "Run folder exists: inspect its status before submitting another job."
    if not os.environ.get("FIREWORKS_API_KEY"):
        os.environ["FIREWORKS_API_KEY"] = getpass("Fireworks API key: ").strip()
    assert os.environ["FIREWORKS_API_KEY"], "No key supplied."
    subprocess.run(command, cwd=HERE, check=True)
else:
    print("Training off. No job submitted.")

### Compare your adapter on serverless
After training and promotion, optionally load **your own model** using the serverless appendix in
[Skill → Weights](../skill_to_weights/skill_to_weights.ipynb). No dedicated deployment is needed.
Wait until a probe succeeds before enabling evaluation below. Training and evaluation have separate costs.

**Your run below uses DEVELOPMENT cases; the recorded chart uses TEST cases.**
Do not compare their headline scores directly. For a direct model comparison, evaluate both models on the
same cases and settings. Within your run, all three arms use the same development cases.

Start with development cases. Use a fresh final test for a new model-selection study; the recorded test has
already been reused. All arms receive the same task information and use low effort and temperature 0.7.
The evaluator calibrates its judge before scoring. It uses the study's judge, which also filtered targets;
inspect the answers yourself alongside the judge scores.

The default run evaluates **392 development cases × 4 responses × 3 configurations = 4,704 answers**, plus judging and calibration. The recorded test uses eight responses per case. Budget for evaluation separately from training.


In [ ]:
RUN_EVALUATION = False
MODEL = "accounts/YOUR_ACCOUNT/models/YOUR_NEW_MODEL_NAME"
EVAL_DIR = RUN_DIR / "eval-dev"
BASE = "accounts/fireworks/models/glm-5p3-flash"
eval_command = [sys.executable, str(HERE / "evaluate_pack.py"),
    "--pack-dir", str(BUNDLE), "--split", "dev", "--draws", "4",
    "--effort", "low", "--temperature", "0.7", "--no-system-prompt",
    "--arm", f"base={BASE}", "--arm", f"skill={BASE}:skill",
    "--arm", f"adapter={MODEL}", "--out", str(EVAL_DIR)]
if RUN_EVALUATION:
    assert "YOUR_" not in MODEL, "Set the model you trained and loaded on serverless."
    assert not EVAL_DIR.exists(), "Evaluation folder exists; inspect it before spending again."
    if not os.environ.get("FIREWORKS_API_KEY"):
        os.environ["FIREWORKS_API_KEY"] = getpass("Fireworks API key: ").strip()
    assert os.environ["FIREWORKS_API_KEY"], "No key supplied."
    subprocess.run(eval_command, cwd=HERE, check=True)
else:
    print("Evaluation off. No inference calls made.")

In [ ]:
if (EVAL_DIR / "samples.jsonl").exists():
    current = list(map(json.loads, (EVAL_DIR / "samples.jsonl").read_text().splitlines()))
    dev = {r["id"]: r for r in map(json.loads, (BUNDLE / "caveman_dev.jsonl").read_text().splitlines())}
    figure(current, dev, title="Your run · DEVELOPMENT", note="Different cases from the recorded test: do not compare the headline scores directly.",
           path=EVAL_DIR / "comparison.png");
    from IPython.display import Markdown, display
    display(Markdown("### Your development evaluation\n" + (EVAL_DIR / "summary.md").read_text()))
else:
    print("Your evaluation will appear here; recorded results above remain separate.")

<a id="understand-it"></a>
## 3. Understand the teaching data: what the skill teaches
Read [the pinned skill](sources/SKILL.md) and its [source/license](sources/SOURCE.md).
The target is terse, useful answers that retain necessary facts.

| Choice | Why it is here |
|---|---|
| Task plus supplied facts | Lets us check whether a short answer preserved the answer |
| Six task families | Covers technical, everyday, multi-part, format, clarity and unknown-answer cases |
| Four teacher candidates | Gives the filter alternatives when one answer is poor |
| Shortest **passing** target | Teaches brevity while requiring the answer to pass the checks |
| No student system prompt | Avoids relying on a hidden style instruction at inference |

### Read a failure
This example illustrates a recorded failure. Use the full evaluation to estimate its frequency.
Read the supplied facts and decide whether the failure is the answer, the checker, or both.

### Follow one answer into the weights
The teacher sees the Caveman skill and a task. We check four candidate answers, then choose the shortest
passing answer for each training task. Tasks with no passing answer contribute no training row.
The chart below shows which task families survived selection. Each bar counts tasks, not candidate answers.

The saved example is especially instructive: two sensible answers were rejected because they omitted
the book title required by a literal check. Read all four. **Would you keep that requirement?**
The selected target reflects the checker as well as the skill. This is why inspecting rejected answers matters.


In [ ]:
from notebook_display import teaching_story
teaching = json.loads((BUNDLE / "teaching_trace.json").read_text())
teaching_story(teaching);


The final record below the candidates is the user/assistant pair passed to training after removing the
neutral system message. The assistant answer is the learning target. The Caveman instructions helped
produce it; those instructions are absent from the training input.

**Before changing the recipe:** decide whether these accepted answers teach the style you want and
whether any family lost too many useful examples. More training would repeat the same selected targets.


In [ ]:
answers(recorded["examples"][1])

<a id="change-it"></a>
## 4. Improve the data before guessing at settings
**Part 2 starts here.** You can stop after the worked example or rebuild a teaching set.

1. Inspect development failures by family.
2. Find the corresponding accepted and rejected teacher answers.
3. Change one data or selection rule. Check it on a small batch first.
4. Train a new adapter and compare on development cases.
5. Freeze the recipe, then evaluate on an untouched final set.

For example, valid refusals were being rejected for their wording, and teachers added unsupported advice.
Repairing the checker and the teacher instructions restored coverage. That is more actionable than simply
asking for a longer training run. Keep a concise-prompt control in your next study to test whether the full
skill adds value beyond “answer concisely.”

The existing builder implements `tasks → targets → filter → pack`. The optional cell below prints a
starting command for a **new** data directory; it never executes it. Generating new data is stochastic and
will not recreate the frozen pack byte for byte. Inspect the builder and costs before running it.

In [ ]:
BUILD_DIR = HERE / "artifacts/my-caveman-data"
print("Optional new-data build (paid; NOT executed):")
print("CAVEMAN_DATA_DIR=" + shlex.quote(str(BUILD_DIR.relative_to(HERE))),
      "CAVEMAN_FAMILIES=v2 CAVEMAN_TARGETS_PER_TASK=4 CAVEMAN_TEACHER_EFFORT=low",
      portable_command([sys.executable, str(HERE / "build_caveman_pack.py"), "all", "--n-tasks", "4500"]))

## What did you learn?
Write a short decision: **adapter, prompt, or improve the data**. Include quality, length, one failure,
and the next change you would test. A useful short answer still needs to be correct.

For another skill, continue with [Skill → Weights](../skill_to_weights/skill_to_weights.ipynb).
Its small starter uses fewer cases, one teacher candidate per case, distinct filtering/evaluation judges,
and a one-epoch rank-16 training plan. It teaches the reusable workflow through a smaller experiment.